# Analisis de sentimientos
Es el proceso automatizado de etiquetar datos de acuerdo al sentimiento del texto como: positivo, negativo, neutro. Es usado en compañías para detectar nuevas percepciones y conocimientos de datos en escala. En su esencia es lenguaje natural de procesamiento, una técnica para clasificar la polaridad de texto segun su sentimiento.

Permite el procesamiento de datos en escala y tiempo real, se puede automatizar esta clasificación para entender directamente como las personas hablan y opinan acerca de un tema en específico, lo que permite a organizaciones:
- Tomar acciones basada en el conocimiento adquerido a partir de los datos.
- Entender como las personas estan hablando de una marca vs competidores 
- Analizar comentarios de encuestas y reseñas de productos 
- Analizar solicitudes o reportes de errores en servicios (tickets) para evitar tasa de cancelación (churn)

# Comandos relevantes para el proyecto en la terminal:
### Instalación y configuración  



In [ ]:
# En la terminal con el env. ya creado, versión usada para este proyecto python=3.11
#(sentiment_analysis)  > conda env export --no-builds > environment.yml 
#(sentiment_analysis)  > 
#(sentiment_analysis)  > conda install -c conda-forge transformers
# Instalar
#(sentiment_analysis)  > conda install -c conda-forge ipykernel
# Registrar
#(sentiment_analysis)  > python -m ipykernel install --user --name sentiment_analysis --display-name "Python (sentiment_analysis)" 
#CPU
#(sentiment_analysis)  > conda install pytorch torchvision torchaudio cpuonly -c pytorch
#GPU
#(sentiment_analysis)  > conda install pytorch torchvision torchaudio pytorch-cuda=12.1 -c pytorch -c nvidia
# Forge
#(sentiment_analysis)  > conda install -c conda-forge ipywidgets pandas

In [1]:
import pandas as pd
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer as sia
import matplotlib.pyplot as plt
from deep_translator import GoogleTranslator as ggt
import seaborn as sns
from wordcloud import WordCloud as wc
from textblob import TextBlob
import spacy  
from transformers import pipeline
import pyodbc       # DB API 2.0  
import sqlalchemy   # ORM 

Conexion a SQL usando pyodbc 

In [2]:
#import torch
#import transformers
#print(torch.__version__)
#print(transformers.__version__)
import pyodbc
conexion = pyodbc.connect(
    "DRIVER={ODBC Driver 18 for SQL Server};"
    "SERVER=localhost;"
    "DATABASE=sentiment_analysis_db;"
    "Trusted_Connection=yes;"    
    "TrustServerCertificate=yes;"
)

df_sql = pd.read_sql("SELECT * FROM comentarios;",conexion)

conexion.close()

print(df_sql.shape)
print(df_sql.head())

#with pd.option_context('display.max_rows', None, 'display.max_columns', None, 'display.max_colwidth', None): 
#    display(df_sql)

comentarios = df_sql.iloc[:,1:]

(203, 4)
   id                                              texto  sentimiento  \
0   1  Excelente atención y muy buena disposición en ...          1.0   
1   2  Los productos son de alta calidad y siempre ll...          1.0   
2   3  Me encanta el diseño y la funcionalidad de la ...          1.0   
3   4  Estoy muy contento con el producto, superó mis...          1.0   
4   5  El servicio postventa fue excelente, resolvier...          1.0   

   categoria  
0          1  
1          1  
2          1  
3          1  
4          1  


C:\Users\LENOVO\AppData\Local\Temp\ipykernel_1908\2239844246.py:14: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_sql = pd.read_sql("SELECT * FROM comentarios;",conexion)


Limpieza del dataset especifico para 'comentarios.csv'
1. with open() -> Manejar recursos eficientemente al cerrar
2. Condicionalmente se seleccionan las lineas/observaciones que empiezan con : "
3. Unión de toda la columna 'texto' usando slicing separando los ultimos dos columnas/variables
4. Creación y agregación de los objetos: 'filas' y 'columnas'

In [ ]:
filas = []

#Limpieza
# Se utiliza "latin-1" debido a que el archivo posee caracteres como tildes (á, é etc) o la letra eñe (ñ) que el tradicional utf-8 no soporta 
with open('comentarios.csv',encoding='latin1' ) as f:   # with: Context manager, cerrar el archivo 
    header = next(f)         # Salta header (primera columna de f)
    
    for linea in f:
        # .strip -> elimina espacios blanco y \n  
        linea = linea.strip()

        if linea.startswith('"'):
            linea = linea.strip(',')
            linea = linea.strip('"')
        else:
            linea = linea.strip(',')    

        parte_linea = linea.split(';')      #.split(';') separa en pedazos
        texto = ';'.join(parte_linea[:-2])          # parte_linea[:-2] desde el inicio hasta el penultimo elemento, todo excepto los ultimos 2 elementos
        #texto = ';'    
        sentimiento = parte_linea[-2].strip()
        categoria = parte_linea[-1].strip()
        filas.append([texto,sentimiento,categoria])

comentarios = pd.DataFrame(filas, columns=['texto', 'sentimiento', 'categoria'])
comentarios['sentimiento'] = pd.to_numeric(comentarios['sentimiento'], errors='coerce') # default error raise -> Exception 
comentarios['categoria']   = pd.to_numeric(comentarios['categoria'],   errors='coerce') # Invalid converts into Nan

#comentarios


#with pd.option_context('display.max_rows', None, 'display.max_columns', None, 'display.max_colwidth', None): 
#    display(comentarios)

comentarios.to_csv('comentarios_limpio.csv',index=True, encoding='utf-8-sig',sep='|' )

### TextBlob
Librerías
1. ggt -> Traductor de Google
2. TextBlob: Analisis sencillo, gran velocidad y facilidad. Disponible para idioma inglés (integración con ggt).

In [7]:
traductor = ggt(source='auto',target='en')

polaridad = []
subjetividad = []
sentimiento_tb = []

for texto in comentarios['texto']:
    traduccion = traductor.translate(texto)

    blob = TextBlob(traduccion)
    pol = blob.sentiment.polarity           # Valor númerico float [-1.0 , 1.0] Polaridad del texto negativo - positivo
    sub = blob.sentiment.subjectivity       # Valor númerico float [0.0 , 1.0]  Opinión personal: 0.0 -> hecho factico , 1.0 -> Opinión comentario

    if pol > 0:
        etiqueta = "Positivo"
    elif pol < 0:
        etiqueta = "Negativo"
    else:
        etiqueta = "Neutro"
    
    polaridad.append(pol)
    subjetividad.append(sub)
    sentimiento_tb.append(etiqueta)

comentarios['tb_polaridad'] = polaridad
comentarios['tb_subjetividad'] = subjetividad
comentarios['tb_sentimiento'] = sentimiento_tb

print(comentarios['tb_sentimiento'].value_counts())
comentarios.head(10)

tb_sentimiento
Positivo    115
Negativo     67
Neutro       21
Name: count, dtype: int64


,texto,sentimiento,categoria,tb_polaridad,tb_subjetividad,tb_sentimiento,vader_compound,vader_sentimiento
0,Excelente atención y muy buena disposición en ...,1.0,1,0.955000,0.890000,Positivo,0.8268,Positivo
1,Los productos son de alta calidad y siempre ll...,1.0,1,0.405000,0.770000,Positivo,0.4754,Positivo
2,Me encanta el diseño y la funcionalidad de la ...,1.0,1,0.466667,0.716667,Positivo,0.7964,Positivo
3,"Estoy muy contento con el producto, superó mis...",1.0,1,1.000000,1.000000,Positivo,0.6115,Positivo
4,"El servicio postventa fue excelente, resolvier...",1.0,1,1.000000,1.000000,Positivo,0.4767,Positivo
5,"Muy mala experiencia con la compra, el product...",0.0,1,-0.655000,0.733333,Negativo,-0.7841,Negativo
6,El servicio al cliente fue poco profesional y ...,0.0,1,0.000000,0.000000,Neutro,-0.6688,Negativo
7,"Me siento decepcionado, la calidad no es como ...",0.0,1,-0.750000,0.750000,Negativo,-0.6381,Negativo
8,Tardaron demasiado en responder y no soluciona...,0.0,1,-0.050000,0.400000,Negativo,0.1695,Positivo
9,"No recomendaría este servicio, tuve problemas ...",0.0,1,0.000000,0.000000,Neutro,-0.5873,Negativo


### Vader
Librerías:
1. SentimentIntensityAnalyzer Vader as sia -> Textos informales, emojis, rrss, reseñas. Posee reglas heurísticas para lenguaje informal
2. ggt

In [8]:
analyzer = sia()
traductor = ggt(source='auto', target='en')

compound_scores = []
sentimiento_vader = []

for texto in comentarios['texto']:
    # 1. Traducir al inglés
    traduccion = traductor.translate(texto)
    
    # 2. Analizar con VADER
    scores = analyzer.polarity_scores(traduccion)
    compound = scores['compound']                   # Valor compuesto: pos, neu, neg    -> [-1.0 , 1.0]
    
    # 3. Clasificar con umbrales estándar de VADER
    if compound >= 0.05:    
        etiqueta = "Positivo"
    elif compound <= -0.05:
        etiqueta = "Negativo"
    else:
        etiqueta = "Neutro"

    compound_scores.append(compound)
    sentimiento_vader.append(etiqueta)

comentarios['vader_compound']    = compound_scores
comentarios['vader_sentimiento'] = sentimiento_vader

print(comentarios['vader_sentimiento'].value_counts())
comentarios.head(5)

vader_sentimiento
Positivo    106
Negativo     81
Neutro       16
Name: count, dtype: int64


,texto,sentimiento,categoria,tb_polaridad,tb_subjetividad,tb_sentimiento,vader_compound,vader_sentimiento
0,Excelente atención y muy buena disposición en ...,1.0,1,0.955000,0.890000,Positivo,0.8268,Positivo
1,Los productos son de alta calidad y siempre ll...,1.0,1,0.405000,0.770000,Positivo,0.4754,Positivo
2,Me encanta el diseño y la funcionalidad de la ...,1.0,1,0.466667,0.716667,Positivo,0.7964,Positivo
3,"Estoy muy contento con el producto, superó mis...",1.0,1,1.000000,1.000000,Positivo,0.6115,Positivo
4,"El servicio postventa fue excelente, resolvier...",1.0,1,1.000000,1.000000,Positivo,0.4767,Positivo


### Spacy
Librerías:
1. Spacy ("versión para español") -> Eliminar ruido gramatical 

In [7]:
import spacy
nlp = spacy.load("es_core_news_sm")

polaridad_sp = []
subjetividad_sp = []
sentimiento_sp = []

for texto in comentarios["texto"]:
    # 
    doc = nlp(texto)

    lemma = " ".join([token.lemma_ for token in doc
                        if not token.is_stop and not token.is_punct])

    blob = TextBlob(lemma)
    pol = blob.sentiment.polarity
    sub = blob.sentiment.subjectivity

    if pol > 0:
        etiqueta = "Positivo"
    elif pol < 0:
        etiqueta = "Negativo"
    else:
        etiqueta = "Neutro"

    
    polaridad_sp.append(pol)
    subjetividad_sp.append(sub)
    sentimiento_sp.append(etiqueta)

comentarios["spacy_polaridad"] = polaridad_sp
comentarios["spacy_subjetividad"] = subjetividad_sp
comentarios["spacy_sentimiento"] = sentimiento_sp

print(comentarios['spacy_sentimiento'].value_counts())




c:\Users\LENOVO\anaconda3\envs\sentiment_analysis\Lib\site-packages\spacy\util.py:971: UserWarning: [W095] Model 'es_core_news_sm' (3.7.0) was trained with spaCy v3.7.0 and may not be 100% compatible with the current version (3.8.14). If you see errors or degraded performance, download a newer compatible model or retrain your custom model with the current spaCy version. For more details and available updates, run: python -m spacy validate
  warnings.warn(warn_msg)


spacy_sentimiento
Neutro      178
Positivo     23
Negativo      2
Name: count, dtype: int64


Librerías:
1. Time
2. Transformers -> Pipeline

In [11]:
import time

clasificador = pipeline(
    "sentiment-analysis",
    model="nlptown/bert-base-multilingual-uncased-sentiment"
)

inicio = time.time()
resultado = clasificador(comentarios['texto'][0])
fin = time.time()

print(resultado)
print(f"Tiempo por frase {fin - inicio:.2f} segundos")

Device set to use cpu


[{'label': '5 stars', 'score': 0.7042403221130371}]
Tiempo por frase 0.21 segundos


In [14]:
puntuaciones = []
estrellas = []
seguridad_modelo = []

for texto in comentarios["texto"]:
    resultado = clasificador(texto)[0]
    label = resultado['label']
    stars = int(label[0])
    score = resultado['score']

    if stars <= 2:
        puntuacion = "Negativo"
    elif stars == 3:
        puntuacion = "Neutro"
    else:
        puntuacion = "Positivo"

puntuaciones.append(puntuacion)
estrellas.append(stars)
seguridad_modelo.append(score)

print(comentarios.value_counts())
comentarios["Puntuación"] = puntuaciones
comentarios["Estrellas"] = estrellas
comentarios["seguridad_modelo"] = seguridad_modelo

    #print(f"Resultado: {resultado}")
    #print(f"Frase: {texto}")
    #print(f"Estrellas: {stars}")
    #print(f"Puntuacion: {puntuacion}")
    #print()

texto                                                                                                                                                    sentimiento  categoria  tb_polaridad   tb_subjetividad  tb_sentimiento  vader_compound  vader_sentimiento  spacy_polaridad  spacy_subjetividad  spacy_sentimiento
La exposición de arte moderno en el museo fue maravillosa, cada obra estaba llena de vida y creatividad, un deleite para los amantes del arte.           1.0          2           5.166667e-01  0.616667         Positivo         0.9274         Positivo           0.00             0.0                 Neutro               2
El concierto de la orquesta sinfónica fue una experiencia impresionante, cada pieza interpretada con una pasión que llegaba al alma, lo disfruté mucho.  1.0          2           5.866667e-01  0.653333         Positivo         0.8750         Positivo           0.00             0.0                 Neutro               2
La festividad cultural de este año fue realme

ValueError: Length of values (1) does not match length of index (203)

## Bibliografía:
1. https://huggingface.co/blog/sentiment-analysis-python
2. https://vadersentiment.readthedocs.io/en/latest/pages/about_the_scoring.html  
3. 